# diff

> Word-level diff computation with whitespace preservation.

In [ ]:
#| default_exp diff

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re
from difflib import SequenceMatcher
from typing import List, Dict, Any

## Tokenization

Before we can diff two texts, we need to split them into tokens.
We diff at the **word level** (not character level) so that changes are meaningful units.
Crucially, we preserve whitespace as separate tokens so the reconstructed text is identical to the original.

In [ ]:
#| export
def tokenize_preserving_whitespace(text: str) -> List[str]:
    """Split text into tokens (words and whitespace) preserving all characters."""
    tokens = re.split(r'(\s+)', text)
    return [t for t in tokens if t]

In [ ]:
assert tokenize_preserving_whitespace("hello world") == ["hello", " ", "world"]
assert tokenize_preserving_whitespace("hello  world") == ["hello", "  ", "world"]
assert tokenize_preserving_whitespace("  leading") == ["  ", "leading"]
assert tokenize_preserving_whitespace("") == []
assert "".join(tokenize_preserving_whitespace("hello\nworld  foo")) == "hello\nworld  foo"

## Computing the diff

We use Python's `difflib.SequenceMatcher` to compare the two token lists.
It returns opcodes: `equal`, `insert`, `delete`, `replace`.

We convert these into our op format:
- `{"type": "equal", "text": "..."}` — unchanged text
- `{"type": "add", "text": "..."}` — new text inserted
- `{"type": "remove", "text": "..."}` — old text deleted
- `{"type": "replace", "old_text": "...", "new_text": "..."}` — text swapped

In [ ]:
#| export
def compute_diff(original: str, rewritten: str) -> List[Dict[str, Any]]:
    """Compute word-level diff between original and rewritten text.
    
    Returns a list of ops: equal, add, remove, replace.
    This is the raw diff — no cleanup or grouping applied."""
    if not original and not rewritten:
        return [{"type": "equal", "text": ""}]
    if not original:
        return [{"type": "add", "text": rewritten}]
    if not rewritten:
        return [{"type": "remove", "text": original}]

    A = tokenize_preserving_whitespace(original)
    B = tokenize_preserving_whitespace(rewritten)
    # autojunk=False is essential: with the default (True), SequenceMatcher
    # treats any token in >1% of a sequence longer than 200 items as "popular
    # junk" and skips it when matching. In prose that hits the space token and
    # common words (the/to/a/and) — exactly the anchors that keep edits
    # word-sized — so a long document collapses into paragraph-sized
    # replacements. Disabling autojunk keeps changes word-level on any length.
    sm = SequenceMatcher(None, A, B, autojunk=False)
    ops = []

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            ops.append({"type": "equal", "text": "".join(A[i1:i2])})
        elif tag == "insert":
            ops.append({"type": "add", "text": "".join(B[j1:j2])})
        elif tag == "delete":
            ops.append({"type": "remove", "text": "".join(A[i1:i2])})
        elif tag == "replace":
            ops.append({"type": "replace", "old_text": "".join(A[i1:i2]), "new_text": "".join(B[j1:j2])})

    return ops

In [ ]:
# Test: identical text -> single equal op
ops = compute_diff("hello world", "hello world")
assert len(ops) == 1
assert ops[0] == {"type": "equal", "text": "hello world"}

In [ ]:
# Test: empty inputs
assert compute_diff("", "") == [{"type": "equal", "text": ""}]
assert compute_diff("", "hello") == [{"type": "add", "text": "hello"}]
assert compute_diff("hello", "") == [{"type": "remove", "text": "hello"}]

In [ ]:
# Test: simple word replacement
ops = compute_diff("the old cat", "the new cat")
assert ops[0] == {"type": "equal", "text": "the "}
assert ops[1] == {"type": "replace", "old_text": "old", "new_text": "new"}
assert ops[2] == {"type": "equal", "text": " cat"}

In [ ]:
# Test: insertion
ops = compute_diff("I like cats", "I really like cats")
assert any(op["type"] == "add" and "really" in op["text"] for op in ops)

In [ ]:
# Test: deletion
ops = compute_diff("I really like cats", "I like cats")
assert any(op["type"] == "remove" and "really" in op["text"] for op in ops)

In [ ]:
# Test: multiline
ops = compute_diff("line one\nline two", "line one\nline three")
assert ops[0]["type"] == "equal"
assert any(op["type"] == "replace" for op in ops)

# Test: reconstructing original and rewritten from ops
def reconstruct(ops):
    """Helper to verify ops can reconstruct both texts."""
    original = ""
    rewritten = ""
    for op in ops:
        if op["type"] == "equal":
            original += op["text"]
            rewritten += op["text"]
        elif op["type"] == "remove":
            original += op["text"]
        elif op["type"] == "add":
            rewritten += op["text"]
        elif op["type"] == "replace":
            original += op["old_text"]
            rewritten += op["new_text"]
    return original, rewritten

orig, rewr = reconstruct(compute_diff("the old cat sat", "the new cat lay"))
assert orig == "the old cat sat"
assert rewr == "the new cat lay"

In [ ]:
# Test: long input stays word-level (autojunk regression)
# On sequences >200 tokens, SequenceMatcher's default autojunk drops common
# tokens (spaces, the/to/a) from matching, which used to collapse long diffs
# into paragraph-sized replacements. Here the two texts differ only by a single
# word, so the diff must stay tiny no matter how long the surrounding text is.
import re

_base = ("The system processes the data and returns a result to the user. " * 30).strip()
_orig = _base + " The final value is correct."
_rewr = _base + " The final value is wrong."

ops = compute_diff(_orig, _rewr)
changes = [op for op in ops if op["type"] in ("add", "remove", "replace")]
# Exactly one change, and it must be the single swapped word — not a big block.
assert len(changes) == 1, f"expected 1 change, got {len(changes)}: {changes}"
ch = changes[0]
assert ch["type"] == "replace"
assert len(ch["old_text"].split()) == 1 and len(ch["new_text"].split()) == 1, ch
assert (ch["old_text"], ch["new_text"]) == ("correct.", "wrong.")

# And reconstruction still holds on the long input.
orig2, rewr2 = reconstruct(ops)
assert orig2 == _orig and rewr2 == _rewr

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()